# 07d — Survival Model

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
from scorelab import tuning as T, explain as E, survival as SV, causal as CA
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Whether modelling *when* a loan defaults adds anything to the card's *whether*. A discrete-time hazard (multinomial logit by months on book, with prepayment as a competing risk) is fitted on the training vintages with the card's WOE attributes as covariates. Its cumulative incidence of default at month 24 is the probability of the label event, so it compares directly with the card's PD on 2016; its curves show *how* each score band's defaults accumulate.

Timing comes from the same reconstruction as the label (90+ DPD = last payment + 4 months), so the hazard inherits that approximation: it cannot see a cure, and a charge-off's timing is only as good as its last payment date.

Decisions: **AA — the hazard specification** (baseline bands, covariates, prepayment treatment), **AB — what the comparison means for the card**.

In [ ]:
frame = SV.event_times(SV.event_frame(con, load_frame(include_indeterminate=True)))
card, specs = S.load_card(MODELS / "champion"); card_attrs = card.attributes
cov = [a + "_woe" for a in card_attrs]
frame = frame.join(B.transform_woe(frame, {a: specs[a] for a in card_attrs}))
tr, oot = frame[frame.split == "train"], frame[frame.split == "oot"]
display(pd.crosstab(frame.label, frame.exit_type, margins=True))      # default exactly when label == bad

In [ ]:
# Evidence — observed cumulative incidence of default and prepayment by vintage (Aalen-Johansen)
emp_v = SV.empirical_cif(frame, group="vintage_year")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for y, d in emp_v.groupby("vintage_year"):
    axes[0].plot(d.month, d.cif_default, label=y); axes[1].plot(d.month, d.cif_prepay, label=y)
axes[0].set_title("cumulative default (90+ DPD)"); axes[1].set_title("cumulative prepayment (paid in full)")
for ax in axes: ax.set_xlabel("months on book"); ax.legend()
plt.show()

### Decision AA — Hazard specification

Baseline: piecewise-constant in 3-month bands (the default) or monthly? Covariates: the card's WOE attributes only (a like-for-like comparison) or more? Prepayment: a competing risk (the CIF then matches the label) or censoring (the "if nobody prepaid" default probability, always higher)? How many loans to expand (each contributes up to 24 rows)?

**Your reasoning:**

_(write here)_

In [ ]:
HAZARD_SAMPLE = 150_000                     # training loans expanded to person-months
BANDS = SV.month_bands((3, 6, 9, 12, 15, 18, 21, 24))
trs = tr.sample(min(HAZARD_SAMPLE, len(tr)), random_state=42)
pp = SV.person_period(trs, cov)
print(f"{len(trs):,} loans -> {len(pp):,} person-months; outcomes:", pp.outcome.value_counts().to_dict())
hm = SV.HazardModel(cov, bands=BANDS).fit(pp)
base, coefs = hm.baseline_table(); display(base); display(coefs)

In [ ]:
# The comparison on 2016 (labelled loans, as everywhere else): 24-month CIF vs the card's PD
lab = oot[oot.label.isin(["bad", "good"])]
cif = hm.cumulative_incidence(lab); cif_cens = hm.cumulative_incidence(lab, competing=False)
p_card = S.score(lab, card, specs)["p_bad"].to_numpy()
cmp = V.model_comparison(lab["bad"], {"Scorecard (champion)": p_card, "Hazard, CIF at 24 (prepay competing)": cif[:, -1],
                                      "Hazard, prepay as censoring": cif_cens[:, -1]}, reference="Scorecard (champion)")
display(cmp.drop(columns=["fit_seconds", "explainability"]))
from scipy.stats import spearmanr
print("rank correlation, card PD vs hazard CIF(24): %.3f" % spearmanr(p_card, cif[:, -1]).correlation)
print("mean predicted: card %.4f · CIF %.4f · censored %.4f · observed %.4f" % (p_card.mean(), cif[:, -1].mean(), cif_cens[:, -1].mean(), lab["bad"].mean()))
cmp.to_csv(config.DOCS_DIR / "exhibits" / "survival_vs_card_oot.csv", index=False)

In [ ]:
# Evidence — cumulative default curve by card score band: predicted (hazard) vs observed (Aalen-Johansen), 2016
oot_s = oot.assign(score=S.score(oot, card, specs)["score"])
oot_s["band"] = pd.qcut(oot_s.score.rank(method="first"), 5, labels=["1 riskiest", "2", "3", "4", "5 safest"])
pred = SV.predicted_cif_by_group(hm.cumulative_incidence(oot_s), oot_s["band"])
emp = SV.empirical_cif(oot_s, group="band")
fig, ax = plt.subplots(figsize=(9, 5))
for i, (b, d) in enumerate(emp.groupby("band", observed=True)):
    ax.plot(d.month, d.cif_default, c=f"C{i}", label=f"band {b} observed")
    p = pred[pred.band == b]; ax.plot(p.month, p.cif_default_pred, c=f"C{i}", ls="--")
ax.set_xlabel("months on book"); ax.set_ylabel("cumulative default (90+ DPD)"); ax.set_title("2016 by card score quintile: observed (solid) vs hazard (dashed)")
ax.legend(fontsize=8); plt.show()
emp.merge(pred, on=["band", "month"], how="left").to_csv(config.DOCS_DIR / "exhibits" / "cif_by_score_band_oot.csv", index=False)

### Decision AB — What the comparison means for the card

Does the hazard rank 2016 better or worse than the card (same attributes, different target construction)? Is the CIF at 24 months better calibrated to 2016 than the card's PD (the card was fitted on 2012–2014 bad rates)? Do the score bands separate early (months 6–12) or only late? Would a hazard model change any decision, or is its value monitoring (early-month incidence by band as a leading indicator for NB09)?

**Your reasoning:**

_(write here)_